[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astheeggeggs/testing_colab/blob/main/LDSC_Practical_2_Colab.ipynb)

# LDSC Practical 2 — Genetic Correlation (Colab version)

**MadhurBain Singh, Benjamin Neale**  
**March 5th, 2025**  
**Adapted from Andrew Grotzinger, March 2023**

This notebook is a Colab-friendly rewrite of the original R practical.
It assumes an **R runtime** in Colab.

**Before you start**
- Put the practical files in one public Google Drive folder, or upload them into `/content`.
- Make sure the folder contains the `EUR/` and `EAS/` directories used below.
- Run the setup cells from top to bottom.

## 1) Environment setup / required packages

This notebook uses an R runtime.
We install the packages needed for the practical from CRAN and GitHub.

Packages used here:
- `data.table`
- `dplyr`
- `remotes`
- `GenomicSEM`
- `corrplot`

In [1]:
# System libraries that are often useful for R package installation in Colab
system("apt-get update -qq")
system("apt-get install -y -qq libcurl4-openssl-dev libxml2-dev libssl-dev")
system('python -m pip -q install gdown')
system('gdown --folder "https://drive.google.com/drive/folders/1L0C2xpYVVSz11n49bKb8ivCzc70Lmbyc?usp=sharing" -O /content/LDSC_Practical_2')

In [2]:
# Install CRAN packages
install.packages(
  c("data.table", "dplyr", "remotes", "corrplot"),
  repos = "https://cloud.r-project.org"
)

# Install GenomicSEM from GitHub
remotes::install_github("GenomicSEM/GenomicSEM", upgrade = "never")

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)


Installing 18 packages: R.methodsS3, iterators, R.oo, foreach, quadprog, numDeriv, pbivnorm, mnormt, gtools, proxy, mgsub, R.utils, splitstackshape, doParallel, lavaan, gdata, e1071, plyr

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



── R CMD build ─────────────────────────────────────────────────────────────────
* checking for file ‘/tmp/RtmpcU6o7f/remotese4b64186879/GenomicSEM-GenomicSEM-123ec96/DESCRIPTION’ ... OK
* preparing ‘GenomicSEM’:
* checking DESCRIPTION meta-information ... OK
* excluding invalid files
Subdirectory 'man' contains invalid file names:
  ‘decisiontree.png’
* checking for LF line-endings in source and make files and shell scripts
* checking for empty or unneeded directories
Omitted ‘LazyData’ from DESCRIPTION
* building ‘GenomicSEM_0.0.5.tar.gz’



Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



### Load packages

In [3]:
library(data.table)
library(dplyr)
library(GenomicSEM)
library(corrplot)

sessionInfo()


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%



Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“replacing previous import ‘gdata::nobs’ by ‘lavaan::nobs’ when loading ‘GenomicSEM’”
Warning message:
“replacing previous import ‘gdata::last’ by ‘data.table::last’ when loading ‘GenomicSEM’”
Warning message:
“replacing previous import ‘gdata::first’ by ‘data.table::first’ when loading ‘GenomicSEM’”
Warning message:
“replacing previous import ‘gdata::resample’ by ‘R.utils::resample’ when loading ‘GenomicSEM’”
Warning message:
“replacing previous import ‘gdata::env’ by ‘R.utils::env’ when loading ‘GenomicSEM’”
Warning message:
“replacing previous import ‘data.table::first’ by ‘dplyr::f

R version 4.6.0 (2026-04-24)
Platform: x86_64-pc-linux-gnu
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/openblas-pthread/libblas.so.3 
LAPACK: /usr/lib/x86_64-linux-gnu/openblas-pthread/libopenblasp-r0.3.20.so;  LAPACK version 3.10.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Etc/UTC
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] corrplot_0.95     GenomicSEM_0.0.5  dplyr_1.2.1       data.table_1.18.4

loaded via a namespace (and not attached):
 [1] generics_0.1.4          class_7.3-23            gtools

In [27]:
system("python -m pip -q install gdown")
system('gdown "https://drive.google.com/drive/folders/1L0C2xpYVVSz11n49bKb8ivCzc70Lmbyc" --folder -O /content/LDSC_Practical_2', intern=TRUE)

Warning message in system("gdown \"https://drive.google.com/drive/folders/1L0C2xpYVVSz11n49bKb8ivCzc70Lmbyc\" --folder -O /content/LDSC_Practical_2", :
“running command 'gdown "https://drive.google.com/drive/folders/1L0C2xpYVVSz11n49bKb8ivCzc70Lmbyc" --folder -O /content/LDSC_Practical_2' had status 1”


[1] "Retrieving folder 1t-RUKTwnNNEGNILkjy2KjhE5r9ZKaWVb EAS"           
 [2] "Retrieving folder 1J3KJplRazob-tvLK9iAD48MZ2I2ppPWN eas_ldscores"  
 [3] "Processing file 1uBGjhQOqbCIXxGNeEKrZp_Hg7BNoj5gJ 1.l2.ldscore.gz" 
 [4] "Processing file 1HHIpXVIKlpMNl6ekEUKBfUpSwbbQrPQ7 1.l2.M"          
 [5] "Processing file 1Io1jWYLpzBaHWkhgahuDA3W6mAUd46Iz 1.l2.M_5_50"     
 [6] "Processing file 1cksak79x0opG0APcZEYdUPjr_SV7_3vR 2.l2.ldscore.gz" 
 [7] "Processing file 1U3fo3cKvoJAxmsb_MXu7e1UzHp--KgHI 2.l2.M"          
 [8] "Processing file 1bbLglpTqFJOrVz06NXJzaZo7dHEZxlou 2.l2.M_5_50"     
 [9] "Processing file 1mTQ1VuFvPzpFRISxfRc132G71TjqexKh 3.l2.ldscore.gz" 
[10] "Processing file 1Ow16xptXjuOzTTBI-OF2G_W4n0F_NHx8 3.l2.M"          
[11] "Processing file 1YgxH63A2T9tKlrHEWqx59ZudJutn8Gc6 3.l2.M_5_50"     
[12] "Processing file 1qNk0wxHkPyKp33XzYZY0i7qIn1bJBkc5 4.l2.ldscore.gz" 
[13] "Processing file 1NdIOR-WEBtHbUk9b7myH_lz3GALPO8JD 4.l2.M"          
[14] "Processing file 1ti7cpf8Lpuh5ZZrZJpjRcsh2GuQ4XnWV 4.l2.M_5_50"     
[15] "Processing file 14Md1blKUsWyhQFL0aI384GvfQzUHc8d0 5.l2.ldscore.gz" 
[16] "Processing file 1BHWTEUyAkpycX7XHZmc4SAO_xlnEyqln 5.l2.M"          
[17] "Processing file 1339B8UXyK2klim7ndaT7rHkx4hHy-MLG 5.l2.M_5_50"     
[18] "Processing file 1aA_xSFJT_rLM9sSiFRBrd7Qay50Q8iyH 6.l2.ldscore.gz" 
[19] "Processing file 1rjOkRcAHGnkD4pO4JSs_Z9t9j3UjnBco 6.l2.M"          
[20] "Processing file 1MsmEo9G9OEZQdOme_7llb8mClL3C6Ta7 6.l2.M_5_50"     
[21] "Processing file 1AAZB2UvOy6XbFESJ9l-NxcHaIZlemfQj 7.l2.ldscore.gz" 
[22] "Processing file 1-Jbi2ZzmqD06R5e38kX5AZ_sBFa9JxZw 7.l2.M"          
[23] "Processing file 1iWLAHJC0FJburyhnhSiQZJscIt0HykZB 7.l2.M_5_50"     
[24] "Processing file 18QoFoa0EWS6VCdYPXj_Mi7ESkteWZ9To 8.l2.ldscore.gz" 
[25] "Processing file 142I-d2ucR5hyktSA_pL-XDF3BBOcK2uE 8.l2.M"          
[26] "Processing file 1wy2Eelon7idpAVX0st8gNw4wzKNtm6Tk 8.l2.M_5_50"     
[27] "Processing file 1KzEq0YAqDD3uN2aeUbkyFddlSaReA2Y9 9.l2.ldscore.gz" 
[28] "Processing file 1e8MxeGdmn5RFAsC5Bf7Hul-15-Nn38pq 9.l2.M"          
[29] "Processing file 1mmZ1AP0TLz4NEl3IdVucodCCz4zQy1fT 9.l2.M_5_50"     
[30] "Processing file 1biLEfeoSCw4wOgaiLuc5UxPHGjtsnBz1 10.l2.ldscore.gz"
[31] "Processing file 1Zq1usNVC-qmQUtsurVlrKtz36mNvvuWi 10.l2.M"         
[32] "Processing file 1GA9N9L3FFv6aIInKXdK7uyJS_meinToU 10.l2.M_5_50"    
[33] "Processing file 1ldR-kzc13WsRNwG54ZU642Ge0wZpjbOk 11.l2.ldscore.gz"
[34] "Processing file 1VGRtjfZSBQrH8pBYxtT4ez2ys6lTDB5Q 11.l2.M"         
[35] "Processing file 1EVHjVIDYdB5jaQoGoIsXvDp5v7ODz8su 11.l2.M_5_50"    
[36] "Processing file 19NHoP_8La7K3c6GOU8DnLN7L3Kc-1Dfs 12.l2.ldscore.gz"
[37] "Processing file 1Si64-wtHFrHThAK6187JiFmqPcmAZmXe 12.l2.M"         
[38] "Processing file 14tRl08WG_7OhVFO10XV-m1Mg6JO8sXvi 12.l2.M_5_50"    
[39] "Processing file 11UiUxg0nDvH36Salt6M5Sbv6tvKhRJT8 13.l2.ldscore.gz"
[40] "Processing file 12QYWYCnq46mIK0dCv_e8irTC9ZpFtgKr 13.l2.M"         
[41] "Processing file 1O_rmZZo79i9fwUUqHqaFqGAlhd5YsIGO 13.l2.M_5_50"    
[42] "Processing file 1i5N13ZIfdcgrmKqf4Q6GSlDKfy8HUvuH 14.l2.ldscore.gz"
[43] "Processing file 1M4J2POlQ-2cIjjAu3xiHfv0HC7oTiU_F 14.l2.M"         
[44] "Processing file 1mfnB0ZpXKbqnvQKxYbl6RR9Gr3A5G8UH 14.l2.M_5_50"    
[45] "Processing file 1Uy89QboqtCX9pSHNovbJJwdj1w_niaWN 15.l2.ldscore.gz"
[46] "Processing file 1FECfMlGK4MIxobsU2fsJgFFBYgVER2Ux 15.l2.M"         
[47] "Processing file 1XKSR0vmET4ckWupSpPSW1xNTFZOjAAji 15.l2.M_5_50"    
[48] "Processing file 13fSMSlH79ChZLfMkU9EC7H5JJ4ScxkDq 16.l2.ldscore.gz"
[49] "Processing file 1ddy2vr5CNZfNqaLYkSSmAW47vAmhaUpk 16.l2.M"         
[50] "Processing file 143D4Yi7QmsCN2bTwhTKixCNLSiUAC0Fc 16.l2.M_5_50"    
[51] "Processing file 1wVYN6xSC3GEHXheCaRGW7sCtKbaTsEP_ 17.l2.ldscore.gz"
[52] "Processing file 15bgg0MZfl_qg5zl-0N--gHNzcHhAZLqM 17.l2.M"         
attr(,"status")
[1] 1
attr(,"errmsg")
[1] "Resource temporarily unavailable"

In [30]:
system("rm -rf /content/LDSC_Practical_2")
system("mkdir -p /content/LDSC_Practical_2")
system('gdown --folder "https://drive.google.com/drive/folders/1L0C2xpYVVSz11n49bKb8ivCzc70Lmbyc" -O /content/LDSC_Practical_2',intern=TRUE)
system("find /content/LDSC_Practical_2 -maxdepth 4 -type f | head -50",intern=TRUE)

Warning message in system("gdown --folder \"https://drive.google.com/drive/folders/1L0C2xpYVVSz11n49bKb8ivCzc70Lmbyc\" -O /content/LDSC_Practical_2", :
“running command 'gdown --folder "https://drive.google.com/drive/folders/1L0C2xpYVVSz11n49bKb8ivCzc70Lmbyc" -O /content/LDSC_Practical_2' had status 1”


[1] "Retrieving folder 1t-RUKTwnNNEGNILkjy2KjhE5r9ZKaWVb EAS"           
 [2] "Retrieving folder 1J3KJplRazob-tvLK9iAD48MZ2I2ppPWN eas_ldscores"  
 [3] "Processing file 1uBGjhQOqbCIXxGNeEKrZp_Hg7BNoj5gJ 1.l2.ldscore.gz" 
 [4] "Processing file 1HHIpXVIKlpMNl6ekEUKBfUpSwbbQrPQ7 1.l2.M"          
 [5] "Processing file 1Io1jWYLpzBaHWkhgahuDA3W6mAUd46Iz 1.l2.M_5_50"     
 [6] "Processing file 1cksak79x0opG0APcZEYdUPjr_SV7_3vR 2.l2.ldscore.gz" 
 [7] "Processing file 1U3fo3cKvoJAxmsb_MXu7e1UzHp--KgHI 2.l2.M"          
 [8] "Processing file 1bbLglpTqFJOrVz06NXJzaZo7dHEZxlou 2.l2.M_5_50"     
 [9] "Processing file 1mTQ1VuFvPzpFRISxfRc132G71TjqexKh 3.l2.ldscore.gz" 
[10] "Processing file 1Ow16xptXjuOzTTBI-OF2G_W4n0F_NHx8 3.l2.M"          
[11] "Processing file 1YgxH63A2T9tKlrHEWqx59ZudJutn8Gc6 3.l2.M_5_50"     
[12] "Processing file 1qNk0wxHkPyKp33XzYZY0i7qIn1bJBkc5 4.l2.ldscore.gz" 
[13] "Processing file 1NdIOR-WEBtHbUk9b7myH_lz3GALPO8JD 4.l2.M"          
[14] "Processing file 1ti7cpf8Lpuh5ZZrZJpjRcsh2GuQ4XnWV 4.l2.M_5_50"     
[15] "Processing file 14Md1blKUsWyhQFL0aI384GvfQzUHc8d0 5.l2.ldscore.gz" 
[16] "Processing file 1BHWTEUyAkpycX7XHZmc4SAO_xlnEyqln 5.l2.M"          
[17] "Processing file 1339B8UXyK2klim7ndaT7rHkx4hHy-MLG 5.l2.M_5_50"     
[18] "Processing file 1aA_xSFJT_rLM9sSiFRBrd7Qay50Q8iyH 6.l2.ldscore.gz" 
[19] "Processing file 1rjOkRcAHGnkD4pO4JSs_Z9t9j3UjnBco 6.l2.M"          
[20] "Processing file 1MsmEo9G9OEZQdOme_7llb8mClL3C6Ta7 6.l2.M_5_50"     
[21] "Processing file 1AAZB2UvOy6XbFESJ9l-NxcHaIZlemfQj 7.l2.ldscore.gz" 
[22] "Processing file 1-Jbi2ZzmqD06R5e38kX5AZ_sBFa9JxZw 7.l2.M"          
[23] "Processing file 1iWLAHJC0FJburyhnhSiQZJscIt0HykZB 7.l2.M_5_50"     
[24] "Processing file 18QoFoa0EWS6VCdYPXj_Mi7ESkteWZ9To 8.l2.ldscore.gz" 
[25] "Processing file 142I-d2ucR5hyktSA_pL-XDF3BBOcK2uE 8.l2.M"          
[26] "Processing file 1wy2Eelon7idpAVX0st8gNw4wzKNtm6Tk 8.l2.M_5_50"     
[27] "Processing file 1KzEq0YAqDD3uN2aeUbkyFddlSaReA2Y9 9.l2.ldscore.gz" 
[28] "Processing file 1e8MxeGdmn5RFAsC5Bf7Hul-15-Nn38pq 9.l2.M"          
[29] "Processing file 1mmZ1AP0TLz4NEl3IdVucodCCz4zQy1fT 9.l2.M_5_50"     
[30] "Processing file 1biLEfeoSCw4wOgaiLuc5UxPHGjtsnBz1 10.l2.ldscore.gz"
[31] "Processing file 1Zq1usNVC-qmQUtsurVlrKtz36mNvvuWi 10.l2.M"         
[32] "Processing file 1GA9N9L3FFv6aIInKXdK7uyJS_meinToU 10.l2.M_5_50"    
[33] "Processing file 1ldR-kzc13WsRNwG54ZU642Ge0wZpjbOk 11.l2.ldscore.gz"
[34] "Processing file 1VGRtjfZSBQrH8pBYxtT4ez2ys6lTDB5Q 11.l2.M"         
[35] "Processing file 1EVHjVIDYdB5jaQoGoIsXvDp5v7ODz8su 11.l2.M_5_50"    
[36] "Processing file 19NHoP_8La7K3c6GOU8DnLN7L3Kc-1Dfs 12.l2.ldscore.gz"
[37] "Processing file 1Si64-wtHFrHThAK6187JiFmqPcmAZmXe 12.l2.M"         
[38] "Processing file 14tRl08WG_7OhVFO10XV-m1Mg6JO8sXvi 12.l2.M_5_50"    
[39] "Processing file 11UiUxg0nDvH36Salt6M5Sbv6tvKhRJT8 13.l2.ldscore.gz"
[40] "Processing file 12QYWYCnq46mIK0dCv_e8irTC9ZpFtgKr 13.l2.M"         
[41] "Processing file 1O_rmZZo79i9fwUUqHqaFqGAlhd5YsIGO 13.l2.M_5_50"    
[42] "Processing file 1i5N13ZIfdcgrmKqf4Q6GSlDKfy8HUvuH 14.l2.ldscore.gz"
[43] "Processing file 1M4J2POlQ-2cIjjAu3xiHfv0HC7oTiU_F 14.l2.M"         
[44] "Processing file 1mfnB0ZpXKbqnvQKxYbl6RR9Gr3A5G8UH 14.l2.M_5_50"    
[45] "Processing file 1Uy89QboqtCX9pSHNovbJJwdj1w_niaWN 15.l2.ldscore.gz"
[46] "Processing file 1FECfMlGK4MIxobsU2fsJgFFBYgVER2Ux 15.l2.M"         
[47] "Processing file 1XKSR0vmET4ckWupSpPSW1xNTFZOjAAji 15.l2.M_5_50"    
[48] "Processing file 13fSMSlH79ChZLfMkU9EC7H5JJ4ScxkDq 16.l2.ldscore.gz"
[49] "Processing file 1ddy2vr5CNZfNqaLYkSSmAW47vAmhaUpk 16.l2.M"         
[50] "Processing file 143D4Yi7QmsCN2bTwhTKixCNLSiUAC0Fc 16.l2.M_5_50"    
[51] "Processing file 1wVYN6xSC3GEHXheCaRGW7sCtKbaTsEP_ 17.l2.ldscore.gz"
[52] "Processing file 15bgg0MZfl_qg5zl-0N--gHNzcHhAZLqM 17.l2.M"         
attr(,"status")
[1] 1
attr(,"errmsg")
[1] "Resource temporarily unavailable"

character(0)

## 2) Colab working directory and file helpers

The code below assumes that the practical files are available in `/content/LDSC_Practical_2`.
If you used the download cell above, that is where the files will be.

In [28]:
getwd()

[1] "/"

In [24]:
dir()

[1] "bin"               "boot"              "datalab"          
 [4] "dev"               "etc"               "home"             
 [7] "kaggle"            "lib"               "lib32"            
[10] "lib64"             "libx32"            "media"            
[13] "mnt"               "opt"               "proc"             
[16] "python-apt"        "python-apt.tar.xz" "root"             
[19] "run"               "sbin"              "srv"              
[22] "sys"               "tmp"               "tools"            
[25] "usr"               "var"

In [10]:
root_dir <- "LDSC_Practical_2"

# Keep paths explicit rather than relying on setwd()
eur_dir <- file.path(root_dir, "EUR")
eas_dir <- file.path(root_dir, "EAS")

# Helper functions for cleaner file previews in notebooks
preview_text <- function(path, n = 10) {
  cat(readLines(path, n = n), sep = "\n")
  cat("\n")
}

preview_gz <- function(path, n = 10) {
  con <- gzfile(path, open = "rt")
  on.exit(close(con), add = TRUE)
  cat(readLines(con, n = n), sep = "\n")
  cat("\n")
}

count_lines <- function(path) {
  length(readLines(path, warn = FALSE))
}

count_gz_lines <- function(path) {
  con <- gzfile(path, open = "rt")
  on.exit(close(con), add = TRUE)
  length(readLines(con, warn = FALSE))
}

list.files(root_dir, recursive = TRUE)[1:min(20, length(list.files(root_dir, recursive = TRUE)))]

[1] NA

## Part 1: Continuous traits example in European ancestry

This section uses chromosome 1 summary statistics for BMI and height.

### HapMap3 SNPs

As before, we restrict the analyses to high-quality SNPs in the GWAS summary statistics that overlap with the pre-computed LD scores.

In [ ]:
hm3_eur <- file.path(eur_dir, "eur_w_ld_chr", "w_hm3.snplist")

# Take a look at the first few lines
preview_text(hm3_eur)

# Count the number of lines
cat("Number of lines in HM3 file:", count_lines(hm3_eur), "\n")

**Question:** How many SNPs are there in the HapMap3 reference file?

### Step 1: Munge the data

In [ ]:
# Look at the summary statistics files and their columns
preview_text(file.path(eur_dir, "GIANT_UKB_BMI_EUR_chr1.txt"))
preview_text(file.path(eur_dir, "Yengo_Height_EUR_chr1.txt"))

# For multivariate LDSC, specify a vector of file paths with GWAS summary statistics
files_eur <- c(
  file.path(eur_dir, "GIANT_UKB_BMI_EUR_chr1.txt"),
  file.path(eur_dir, "Yengo_Height_EUR_chr1.txt")
)

# Trait names should match the order of the file paths
trait_names_eur <- c("BMI_chr1", "Height_chr1")

# Quality filters
info.filter <- 0.9
maf.filter <- 0.01

munge(
  files = files_eur,
  hm3 = hm3_eur,
  trait.names = trait_names_eur,
  info.filter = info.filter,
  maf.filter = maf.filter
)

### Step 2: Run LDSC

In [ ]:
# Examine the munged summary statistics
preview_gz(file.path(eur_dir, "BMI.sumstats.gz"))
preview_gz(file.path(eur_dir, "Height.sumstats.gz"))

traits_eur <- c(
  file.path(eur_dir, "BMI.sumstats.gz"),
  file.path(eur_dir, "Height.sumstats.gz")
)

ld_eur <- file.path(eur_dir, "eur_w_ld_chr")
wld_eur <- file.path(eur_dir, "eur_w_ld_chr")
sample.prev_eur <- c(NA, NA)
population.prev_eur <- c(NA, NA)
trait_names_ldsc_eur <- c("BMI", "Height")

LDSC_EUR <- ldsc(
  traits = traits_eur,
  sample.prev = sample.prev_eur,
  population.prev = population.prev_eur,
  ld = ld_eur,
  wld = wld_eur,
  trait.names = trait_names_ldsc_eur
)

save(LDSC_EUR, file = file.path(eur_dir, "LDSC_EUR.RData"))

### Inspect the outputs

In [ ]:
# Genetic covariance matrix
LDSC_EUR$S

# Genetic correlation matrix
cov2cor(LDSC_EUR$S)

# Standard errors from the V matrix
k <- nrow(LDSC_EUR$S)
SE <- matrix(0, k, k)
SE[lower.tri(SE, diag = TRUE)] <- sqrt(diag(LDSC_EUR$V))
SE

# Z-statistics
Z <- LDSC_EUR$S / SE
Z

# P-values
P <- 2 * pnorm(Z, lower.tail = FALSE)
P

### Optional: Create an rg heatmap

In [ ]:
rownames(LDSC_EUR$S) <- colnames(LDSC_EUR$S)

corrplot(
  corr = cov2cor(LDSC_EUR$S),
  method = "color",
  addCoef.col = "dark grey",
  add = FALSE,
  bg = "white",
  diag = TRUE,
  outline = TRUE,
  mar = c(0, 0, 2, 0),
  number.cex = 2,
  cl.pos = "b",
  cl.ratio = 0.125,
  cl.align.text = "l",
  cl.offset = 0.2,
  tl.srt = 45,
  tl.pos = "lt",
  tl.offset = 0.2,
  tl.col = "black",
  pch.col = "white",
  addgrid.col = "black",
  xpd = TRUE,
  tl.cex = 1.3,
  is.corr = TRUE,
  title = "European"
)

## Part 2: Continuous traits example in East Asian ancestry

The summary statistics and LD scores for this section are from East Asian ancestry.

In [ ]:
# Look at the files
preview_text(file.path(eas_dir, "BMI_EAS_chr1.txt"))
preview_text(file.path(eas_dir, "Yengo_Height_EAS_chr1.txt"))

files_eas <- c(
  file.path(eas_dir, "BMI_EAS_chr1.txt"),
  file.path(eas_dir, "Yengo_Height_EAS_chr1.txt")
)

hm3_eas <- file.path(eas_dir, "eas_ldscores", "w_hm3.snplist")
preview_text(hm3_eas)
cat("Number of lines in EAS HM3 file:", count_lines(hm3_eas), "\n")

trait_names_eas <- c("BMI", "Height")
N_eas <- c(163835, NA)
info.filter <- 0.9
maf.filter <- 0.01

munge(
  files = files_eas,
  hm3 = hm3_eas,
  trait.names = trait_names_eas,
  N = N_eas,
  info.filter = info.filter,
  maf.filter = maf.filter
)

In [ ]:
preview_gz(file.path(eas_dir, "BMI.sumstats.gz"))
preview_gz(file.path(eas_dir, "Height.sumstats.gz"))

traits_eas <- c(
  file.path(eas_dir, "BMI.sumstats.gz"),
  file.path(eas_dir, "Height.sumstats.gz")
)

ld_eas <- file.path(eas_dir, "eas_ldscores")
wld_eas <- file.path(eas_dir, "eas_ldscores")
sample.prev_eas <- c(NA, NA)
population.prev_eas <- c(NA, NA)
trait_names_ldsc_eas <- c("BMI", "Height")

LDSC_EAS <- ldsc(
  traits = traits_eas,
  sample.prev = sample.prev_eas,
  population.prev = population.prev_eas,
  ld = ld_eas,
  wld = wld_eas,
  trait.names = trait_names_ldsc_eas
)

save(LDSC_EAS, file = file.path(eas_dir, "LDSC_EAS.RData"))

In [ ]:
# Genetic covariance matrix
LDSC_EAS$S

# Compare with EUR if needed
load(file.path(eur_dir, "LDSC_EUR.RData"))
LDSC_EUR$S

# Genetic correlation matrix
cov2cor(LDSC_EAS$S)

# Standard errors
k <- nrow(LDSC_EAS$S)
SE <- matrix(0, k, k)
SE[lower.tri(SE, diag = TRUE)] <- sqrt(diag(LDSC_EAS$V))
SE

# Z-statistics
Z <- LDSC_EAS$S / SE
Z

# P-values
P <- 2 * pnorm(Z, lower.tail = FALSE)
P

In [ ]:
rownames(LDSC_EAS$S) <- colnames(LDSC_EAS$S)

corrplot(
  corr = cov2cor(LDSC_EAS$S),
  method = "color",
  addCoef.col = "dark grey",
  add = FALSE,
  bg = "white",
  diag = TRUE,
  outline = TRUE,
  mar = c(0, 0, 2, 0),
  number.cex = 2,
  cl.pos = "b",
  cl.ratio = 0.125,
  cl.align.text = "l",
  cl.offset = 0.2,
  tl.srt = 45,
  tl.pos = "lt",
  tl.offset = 0.2,
  tl.col = "black",
  pch.col = "white",
  addgrid.col = "black",
  xpd = TRUE,
  tl.cex = 1.3,
  is.corr = TRUE,
  title = "East Asian"
)

## Part 3: Binary (case/control) traits example in European ancestry

In [ ]:
# Files
preview_text(file.path(eur_dir, "SCZ_chr1.txt"))
preview_text(file.path(eur_dir, "BIP_chr1.txt"))

files_bin <- c(
  file.path(eur_dir, "SCZ_chr1.txt"),
  file.path(eur_dir, "BIP_chr1.txt")
)

hm3_bin <- file.path(eur_dir, "eur_w_ld_chr", "w_hm3.snplist")
trait_names_bin <- c("SCZ", "BIP")
N_bin <- c(NA, NA)
info.filter <- 0.9
maf.filter <- 0.01

munge(
  files = files_bin,
  hm3 = hm3_bin,
  trait.names = trait_names_bin,
  N = N_bin,
  info.filter = info.filter,
  maf.filter = maf.filter
)

In [ ]:
preview_gz(file.path(eur_dir, "SCZ.sumstats.gz"))
preview_gz(file.path(eur_dir, "BIP.sumstats.gz"))

traits_bin <- c(
  file.path(eur_dir, "SCZ.sumstats.gz"),
  file.path(eur_dir, "BIP.sumstats.gz")
)

ld_bin <- file.path(eur_dir, "eur_w_ld_chr")
wld_bin <- file.path(eur_dir, "eur_w_ld_chr")
sample.prev_bin <- c(0.5, 0.5)
population.prev_bin <- c(0.01, 0.02)
trait_names_ldsc_bin <- c("SCZ", "BIP")

LDSC_SCZ_BIP <- ldsc(
  traits = traits_bin,
  sample.prev = sample.prev_bin,
  population.prev = population.prev_bin,
  ld = ld_bin,
  wld = wld_bin,
  trait.names = trait_names_ldsc_bin
)

save(LDSC_SCZ_BIP, file = file.path(eur_dir, "LDSC_SCZ_BIP.RData"))

In [ ]:
# Genetic covariance matrix
LDSC_SCZ_BIP$S

# Genetic correlation matrix
cov2cor(LDSC_SCZ_BIP$S)

# Standard errors
k <- nrow(LDSC_SCZ_BIP$S)
SE <- matrix(0, k, k)
SE[lower.tri(SE, diag = TRUE)] <- sqrt(diag(LDSC_SCZ_BIP$V))
SE

# Z-statistics
Z <- LDSC_SCZ_BIP$S / SE
Z

# P-values
P <- 2 * pnorm(Z, lower.tail = FALSE)
P

In [ ]:
rownames(LDSC_SCZ_BIP$S) <- colnames(LDSC_SCZ_BIP$S)

corrplot(
  corr = cov2cor(LDSC_SCZ_BIP$S),
  method = "color",
  addCoef.col = "dark grey",
  add = FALSE,
  bg = "white",
  diag = TRUE,
  outline = TRUE,
  mar = c(0, 0, 2, 0),
  number.cex = 2,
  cl.pos = "b",
  cl.ratio = 0.125,
  cl.align.text = "l",
  cl.offset = 0.2,
  tl.srt = 45,
  tl.pos = "lt",
  tl.offset = 0.2,
  tl.col = "black",
  pch.col = "white",
  addgrid.col = "black",
  xpd = TRUE,
  tl.cex = 1.3,
  is.corr = TRUE,
  title = "SCZ / BIP"
)

## End

You have now run the Colab version of Practical 2 for genetic correlation.